In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [3]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale [0,1] => normalize => [-1, 1]
transform =  transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform) 
testset = CIFAR10(root="./data", train=False, download=True, transform=transform) 

100%|██████████| 170M/170M [13:50<00:00, 205kB/s]    
c:\Users\kisha\anaconda3\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [5]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

### Build the CNN

In [7]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),# kernel size = 2, stride = 2
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),# kernel size = 2, stride = 2
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)# kernel size = 2, stride = 2
            
        ) 
        
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),
            
            nn.Linear(256, 10)
        )
        
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)
        
        return x

In [8]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training CNN

In [10]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0
    
    for imags, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(imags) # forward propagation
        loss = criterion(output, labels)# loss fnx
        loss.backward() # backward propagation
        optimizer.step() # update params
        
        epoch_training_loss += loss.item()
        
    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=0.6320092295060682
epoch=2/10 & loss=0.5285795326999692
epoch=3/10 & loss=0.4369741860596115
epoch=4/10 & loss=0.358418455295017
epoch=5/10 & loss=0.2841767554011796
epoch=6/10 & loss=0.22631693649989412
epoch=7/10 & loss=0.1765181497739785
epoch=8/10 & loss=0.14676926122344744
epoch=9/10 & loss=0.12334287112764537
epoch=10/10 & loss=0.11097277314437891


In [11]:
# Evaluate

correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images, labels in testloader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy: {100 * correct_labels / total_labels}%")

Accuracy: 75.78%
